# Comparative Reinforcement Learning: DQN vs REINFORCE vs A2C
### LunarLander-v3 — Capstone Integrated Notebook

This notebook implements and compares three reinforcement learning algorithms on LunarLander-v3 (8-dim continuous state, 4 discrete actions):

| Method | Update Style | Key Mechanism | Variance / Bias |
|--------|-------------|---------------|------------------|
| **DQN** | Per-step, off-policy | Experience replay + frozen target net | Low variance, some bias |
| **REINFORCE** | End-of-episode, on-policy | Monte Carlo returns + entropy bonus | High variance, zero bias |
| **A2C** | Per-step, on-policy | TD(0) advantage + separate critic | Medium variance, some bias |

All three agents share a common `Agent` base class and a unified `TrainingMetrics` tracker.

## 1. Import Libraries

In [ ]:
import gymnasium as gym
import numpy as np
import random
import time
import copy
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

import matplotlib.pyplot as plt
from IPython.display import clear_output

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 2. Initialize Environment

In [ ]:
ENV_NAME = 'LunarLander-v3'
env = gym.make(ENV_NAME)

STATE_SIZE = env.observation_space.shape[0]   # 8
ACTION_SIZE = env.action_space.n              # 4

print(f'Environment:       {ENV_NAME}')
print(f'State size:        {STATE_SIZE}')
print(f'Action size:       {ACTION_SIZE}')
env.close()

## 3. Base Agent Class

In [ ]:
class Agent:
    """Base agent — provides random action fallback and action-space introspection."""
    def __init__(self, env):
        self.is_discrete = isinstance(env.action_space, gym.spaces.discrete.Discrete)
        if self.is_discrete:
            self.action_size = env.action_space.n
        else:
            self.action_space_low = env.action_space.low
            self.action_space_high = env.action_space.high
            self.action_shape = env.action_space.shape

    def get_action(self, state):
        if self.is_discrete:
            return random.choice(range(self.action_size))
        return np.random.uniform(self.action_space_low, self.action_space_high, self.action_shape)

## 4. Training Metrics

Unified tracker used by all three agents. Records per-episode reward, loss, and episode length with rolling averages.

In [ ]:
class TrainingMetrics:
    """
    Unified per-episode metric tracker for all agent types.
    Accumulates step-level data within an episode, then flushes on end_episode().
    """
    def __init__(self, method_name, rolling_window=100):
        self.method = method_name
        self.window = rolling_window
        self.rewards = []
        self.losses = []          # primary loss (DQN: Huber, REINFORCE: policy, A2C: total)
        self.episode_lengths = []
        self.extras = {}          # method-specific (epsilon, entropy, critic_loss, etc.)
        self._reset_ep()

    def _reset_ep(self):
        self._ep_reward = 0.0
        self._ep_losses = []
        self._ep_steps = 0
        self._ep_extras = {}

    def step(self, reward, loss=None, **extra):
        self._ep_reward += reward
        self._ep_steps += 1
        if loss is not None:
            self._ep_losses.append(loss)
        for k, v in extra.items():
            self._ep_extras.setdefault(k, []).append(v)

    def end_episode(self, **episode_extras):
        self.rewards.append(self._ep_reward)
        self.losses.append(np.mean(self._ep_losses) if self._ep_losses else 0.0)
        self.episode_lengths.append(self._ep_steps)
        for k, vals in self._ep_extras.items():
            self.extras.setdefault(k, []).append(np.mean(vals))
        for k, v in episode_extras.items():
            self.extras.setdefault(k, []).append(v)
        self._reset_ep()

    def rolling(self, data):
        return [np.mean(data[max(0, i-self.window+1):i+1]) for i in range(len(data))]

    def last_avg(self, n=100):
        return np.mean(self.rewards[-n:]) if self.rewards else 0.0

    def print_status(self, ep):
        r = self.rewards[-1]
        avg = self.last_avg()
        l = self.losses[-1]
        s = self.episode_lengths[-1]
        extra_str = ''
        for k in ['epsilon', 'entropy']:
            if k in self.extras and self.extras[k]:
                extra_str += f' | {k}: {self.extras[k][-1]:.4f}'
        print(f'[{self.method}] Ep {ep:4d} | R: {r:7.1f} (avg {avg:7.1f}) | Loss: {l:8.4f} | Steps: {s:4d}{extra_str}')

## 5. Neural Network Architectures

In [ ]:
# ─── DQN Network ───
class DQNNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, action_size)
        )
    def forward(self, x):
        return self.net(x)


# ─── REINFORCE Policy Network ───
class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, action_size)
        )
    def forward(self, x):
        return self.net(x)


# ─── A2C Actor / Critic (separate networks) ───
class ActorNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, action_size)
        )
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                nn.init.constant_(layer.bias, 0.0)
    def forward(self, x):
        return self.net(x)

class CriticNetwork(nn.Module):
    def __init__(self, state_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                nn.init.constant_(layer.bias, 0.0)
    def forward(self, x):
        return self.net(x)

## 6. Deep Q-Network (DQN) Agent

Features: experience replay buffer, frozen target network (soft-updated), ε-greedy exploration with decay, Huber loss, gradient clipping.

In [ ]:
class DeepQAgent(Agent):
    def __init__(self, env, lr=1e-3, gamma=0.99, eps_start=1.0, eps_end=0.01,
                 eps_decay=0.995, buffer_size=50000, batch_size=64,
                 target_update=10, max_grad_norm=1.0):
        super().__init__(env)
        self.state_size = env.observation_space.shape[0]
        self.gamma = gamma
        self.eps = eps_start
        self.eps_end = eps_end
        self.eps_decay = eps_decay
        self.batch_size = batch_size
        self.target_update = target_update
        self.max_grad_norm = max_grad_norm
        self.learn_step = 0

        # Networks
        self.policy_net = DQNNetwork(self.state_size, self.action_size).to(device)
        self.target_net = DQNNetwork(self.state_size, self.action_size).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.loss_fn = nn.SmoothL1Loss()

        # Replay buffer
        self.buffer = deque(maxlen=buffer_size)

        print(f'=== DQN Agent Initialized ===')
        print(f'  Buffer: {buffer_size}, Batch: {batch_size}, Target update: every {target_update} eps')
        print(f'  Epsilon: {eps_start} → {eps_end} (decay {eps_decay})')

    def get_action(self, state):
        if random.random() < self.eps:
            return random.choice(range(self.action_size))
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            q_vals = self.policy_net(state_t)
        return torch.argmax(q_vals, dim=1).item()

    def store(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def train_step(self):
        """Sample a minibatch and do one gradient step. Returns loss or None."""
        if len(self.buffer) < self.batch_size:
            return None

        batch = random.sample(self.buffer, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states_t    = torch.FloatTensor(np.array(states)).to(device)
        actions_t   = torch.LongTensor(actions).to(device)
        rewards_t   = torch.FloatTensor(rewards).to(device)
        next_st_t   = torch.FloatTensor(np.array(next_states)).to(device)
        dones_t     = torch.FloatTensor(dones).to(device)

        # Current Q
        q_values = self.policy_net(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)

        # Target Q (frozen network)
        with torch.no_grad():
            next_q = self.target_net(next_st_t).max(dim=1)[0]
            target = rewards_t + self.gamma * next_q * (1 - dones_t)

        loss = self.loss_fn(q_values, target)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), self.max_grad_norm)
        self.optimizer.step()

        return loss.item()

    def end_episode(self):
        """Decay epsilon + periodically sync target network."""
        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        self.learn_step += 1
        if self.learn_step % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

## 7. REINFORCE (Monte Carlo Policy Gradient) Agent

Features: discounted + normalized returns, entropy bonus for exploration, gradient clipping. Trains once per episode using the full trajectory.

In [ ]:
class ReinforceAgent(Agent):
    def __init__(self, env, lr=1e-3, gamma=0.99, entropy_coeff=0.01, max_grad_norm=0.5):
        super().__init__(env)
        self.state_size = env.observation_space.shape[0]
        self.gamma = gamma
        self.entropy_coeff = entropy_coeff
        self.max_grad_norm = max_grad_norm

        self.model = PolicyNetwork(self.state_size, self.action_size).to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

        # Episode trajectory buffers
        self.log_probs = []
        self.entropies = []
        self.rewards_buffer = []

        print(f'=== REINFORCE Agent Initialized ===')
        print(f'  LR: {lr}, γ: {gamma}, Entropy coeff: {entropy_coeff}')

    def get_action(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        logits = self.model(state_t)
        dist = Categorical(logits=logits)
        action = dist.sample()
        self.log_probs.append(dist.log_prob(action))
        self.entropies.append(dist.entropy())
        return action.item()

    def store_reward(self, reward):
        self.rewards_buffer.append(reward)

    def train_episode(self):
        """Compute discounted returns, normalize, and do one gradient step. Returns (loss, mean_entropy)."""
        # Discounted returns (Monte Carlo)
        returns = []
        G = 0
        for r in reversed(self.rewards_buffer):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)

        # Normalize returns (scale-invariant baselines)
        if returns.std() > 1e-8:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        log_probs = torch.stack(self.log_probs)
        entropies = torch.stack(self.entropies)

        # Policy gradient loss: -log π(a|s) · G  with entropy bonus
        policy_loss = -(log_probs * returns).sum()
        entropy_bonus = -self.entropy_coeff * entropies.sum()
        loss = policy_loss + entropy_bonus

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
        self.optimizer.step()

        mean_ent = entropies.mean().item()
        loss_val = loss.item()

        # Clear trajectory
        self.log_probs = []
        self.entropies = []
        self.rewards_buffer = []

        return loss_val, mean_ent

## 8. Advantage Actor-Critic (A2C) Agent

Features: separate actor/critic networks with independent optimizers, Huber loss on critic, advantage clipping ±10, entropy bonus, gradient clipping, orthogonal init.

In [ ]:
class A2CAgent(Agent):
    def __init__(self, env, actor_lr=1e-3, critic_lr=5e-4, gamma=0.99,
                 entropy_coeff=0.05, max_grad_norm=0.5, advantage_clip=10.0):
        super().__init__(env)
        self.state_size = env.observation_space.shape[0]
        self.gamma = gamma
        self.entropy_coeff = entropy_coeff
        self.max_grad_norm = max_grad_norm
        self.advantage_clip = advantage_clip

        self.actor = ActorNetwork(self.state_size, self.action_size).to(device)
        self.critic = CriticNetwork(self.state_size).to(device)
        self.actor_opt = optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=critic_lr)
        self.critic_loss_fn = nn.SmoothL1Loss()

        print(f'=== A2C Agent Initialized ===')
        print(f'  Actor LR: {actor_lr}, Critic LR: {critic_lr}')
        print(f'  Entropy: {entropy_coeff}, Adv clip: ±{advantage_clip}')

    def get_action(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = self.actor(state_t)
        dist = Categorical(logits=logits)
        action = dist.sample()
        return action.item()

    def train_step(self, state, action, reward, next_state, done):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        next_t  = torch.FloatTensor(next_state).unsqueeze(0).to(device)
        reward_t = torch.FloatTensor([reward]).to(device)

        # ── Critic update ──
        v = self.critic(state_t).squeeze()
        with torch.no_grad():
            v_next = self.critic(next_t).squeeze()
            td_target = reward_t + self.gamma * v_next * (1 - int(done))

        critic_loss = self.critic_loss_fn(v, td_target)
        self.critic_opt.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), self.max_grad_norm)
        self.critic_opt.step()

        # ── Actor update ──
        with torch.no_grad():
            v_updated = self.critic(state_t).squeeze()
            advantage = torch.clamp(td_target - v_updated, -self.advantage_clip, self.advantage_clip)

        logits = self.actor(state_t)
        dist = Categorical(logits=logits)
        log_prob = dist.log_prob(torch.tensor(action).to(device))
        entropy = dist.entropy()
        actor_loss = -(log_prob * advantage) - self.entropy_coeff * entropy

        self.actor_opt.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), self.max_grad_norm)
        self.actor_opt.step()

        return actor_loss.item() + 0.5 * critic_loss.item(), entropy.item()

## 9. Training Functions

Separate training loops for each method, all returning a `TrainingMetrics` object for unified comparison.

In [ ]:
def train_dqn(num_episodes=1000, print_every=50):
    env = gym.make(ENV_NAME)
    agent = DeepQAgent(env)
    metrics = TrainingMetrics('DQN')

    for ep in range(1, num_episodes + 1):
        state, _ = env.reset()
        done = False
        while not done:
            action = agent.get_action(state)
            next_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            agent.store(state, action, reward, next_state, done)
            loss = agent.train_step()
            metrics.step(reward, loss=loss)
            state = next_state

        agent.end_episode()
        metrics.end_episode(epsilon=agent.eps)
        if ep % print_every == 0:
            metrics.print_status(ep)
        if metrics.last_avg() >= 200 and len(metrics.rewards) >= 100:
            print(f'\n*** DQN SOLVED at episode {ep}! Avg: {metrics.last_avg():.1f} ***')
            break

    env.close()
    return metrics, agent


def train_reinforce(num_episodes=2000, print_every=50):
    env = gym.make(ENV_NAME)
    agent = ReinforceAgent(env)
    metrics = TrainingMetrics('REINFORCE')

    for ep in range(1, num_episodes + 1):
        state, _ = env.reset()
        done = False
        while not done:
            action = agent.get_action(state)
            next_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            agent.store_reward(reward)
            metrics.step(reward)
            state = next_state

        loss, ent = agent.train_episode()
        metrics._ep_losses.append(loss)  # inject episode-level loss
        metrics.end_episode(entropy=ent)
        if ep % print_every == 0:
            metrics.print_status(ep)
        if metrics.last_avg() >= 200 and len(metrics.rewards) >= 100:
            print(f'\n*** REINFORCE SOLVED at episode {ep}! Avg: {metrics.last_avg():.1f} ***')
            break

    env.close()
    return metrics, agent


def train_a2c(num_episodes=2000, print_every=50):
    env = gym.make(ENV_NAME)
    agent = A2CAgent(env)
    metrics = TrainingMetrics('A2C')

    for ep in range(1, num_episodes + 1):
        state, _ = env.reset()
        done = False
        while not done:
            action = agent.get_action(state)
            next_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            loss, ent = agent.train_step(state, action, reward, next_state, done)
            metrics.step(reward, loss=loss, entropy=ent)
            state = next_state

        metrics.end_episode()
        if ep % print_every == 0:
            metrics.print_status(ep)
        if metrics.last_avg() >= 200 and len(metrics.rewards) >= 100:
            print(f'\n*** A2C SOLVED at episode {ep}! Avg: {metrics.last_avg():.1f} ***')
            break

    env.close()
    return metrics, agent

## 10. Run All Training

Each method trains independently on its own environment instance.

In [ ]:
print('='*60)
print('TRAINING: DQN')
print('='*60)
dqn_metrics, dqn_agent = train_dqn(num_episodes=1000)

In [ ]:
print('\n' + '='*60)
print('TRAINING: REINFORCE')
print('='*60)
reinforce_metrics, reinforce_agent = train_reinforce(num_episodes=2000)

In [ ]:
print('\n' + '='*60)
print('TRAINING: A2C')
print('='*60)
a2c_metrics, a2c_agent = train_a2c(num_episodes=2000)

## 11. Comparison Plots

In [ ]:
all_metrics = [dqn_metrics, reinforce_metrics, a2c_metrics]
colors = {'DQN': 'steelblue', 'REINFORCE': 'coral', 'A2C': 'seagreen'}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('DQN vs REINFORCE vs A2C — LunarLander-v3', fontsize=16, fontweight='bold')

# ─── 1. Reward ───
ax = axes[0, 0]
for m in all_metrics:
    eps = range(1, len(m.rewards)+1)
    c = colors[m.method]
    ax.plot(eps, m.rewards, alpha=0.12, color=c)
    ax.plot(eps, m.rolling(m.rewards), color=c, lw=2, label=f'{m.method}')
ax.axhline(200, color='green', ls='--', alpha=0.5, label='Solved (200)')
ax.set_title('Episode Reward (100-ep rolling avg)')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.legend(); ax.grid(True, alpha=0.3)

# ─── 2. Loss ───
ax = axes[0, 1]
for m in all_metrics:
    eps = range(1, len(m.losses)+1)
    c = colors[m.method]
    ax.plot(eps, m.rolling(m.losses), color=c, lw=2, label=m.method)
ax.set_title('Training Loss (100-ep rolling avg)')
ax.set_xlabel('Episode'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# ─── 3. Episode Length ───
ax = axes[1, 0]
for m in all_metrics:
    eps = range(1, len(m.episode_lengths)+1)
    c = colors[m.method]
    ax.plot(eps, m.rolling(m.episode_lengths), color=c, lw=2, label=m.method)
ax.set_title('Episode Length (100-ep rolling avg)')
ax.set_xlabel('Episode'); ax.set_ylabel('Steps')
ax.legend(); ax.grid(True, alpha=0.3)

# ─── 4. Entropy (REINFORCE & A2C only) ───
ax = axes[1, 1]
for m in [reinforce_metrics, a2c_metrics]:
    if 'entropy' in m.extras:
        data = m.extras['entropy']
        eps = range(1, len(data)+1)
        c = colors[m.method]
        ax.plot(eps, data, alpha=0.15, color=c)
        ax.plot(eps, m.rolling(data), color=c, lw=2, label=m.method)
if 'epsilon' in dqn_metrics.extras:
    data = dqn_metrics.extras['epsilon']
    eps = range(1, len(data)+1)
    ax2 = ax.twinx()
    ax2.plot(eps, data, color=colors['DQN'], lw=2, ls='--', label='DQN ε')
    ax2.set_ylabel('DQN Epsilon', color=colors['DQN'])
    ax2.legend(loc='upper right')
ax.set_title('Policy Entropy / DQN Epsilon')
ax.set_xlabel('Episode'); ax.set_ylabel('Entropy (nats)')
ax.legend(loc='center right'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Evaluation (Greedy Policy)

In [ ]:
def evaluate_agent(agent, agent_type, num_episodes=30):
    """Run greedy evaluation (no exploration)."""
    eval_env = gym.make(ENV_NAME)
    rewards = []
    for _ in range(num_episodes):
        state, _ = eval_env.reset()
        done = False
        total = 0
        while not done:
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                if agent_type == 'DQN':
                    q = agent.policy_net(state_t)
                    action = torch.argmax(q, dim=1).item()
                elif agent_type == 'REINFORCE':
                    logits = agent.model(state_t)
                    action = torch.argmax(logits, dim=1).item()
                elif agent_type == 'A2C':
                    logits = agent.actor(state_t)
                    action = torch.argmax(logits, dim=1).item()
            state, reward, term, trunc, _ = eval_env.step(action)
            done = term or trunc
            total += reward
        rewards.append(total)
    eval_env.close()
    return rewards


print(f'{"Method":<12} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 50)
for name, agent, atype in [('DQN', dqn_agent, 'DQN'),
                            ('REINFORCE', reinforce_agent, 'REINFORCE'),
                            ('A2C', a2c_agent, 'A2C')]:
    r = evaluate_agent(agent, atype)
    print(f'{name:<12} {np.mean(r):8.1f} {np.std(r):8.1f} {np.min(r):8.1f} {np.max(r):8.1f}')

## 13. Summary Statistics

In [ ]:
print(f'{"Method":<12} {"Episodes":>10} {"Final 100-avg":>15} {"Best 100-avg":>15} {"Solved?":>10}')
print('-' * 65)
for m in all_metrics:
    n = len(m.rewards)
    final = m.last_avg()
    # Best rolling 100-ep average
    best = max(np.mean(m.rewards[max(0,i-99):i+1]) for i in range(len(m.rewards)))
    solved = '✓' if best >= 200 else '✗'
    print(f'{m.method:<12} {n:10d} {final:15.1f} {best:15.1f} {solved:>10}')

## Cleanup

In [ ]:
print('All training and evaluation complete.')